2. Use linear regression with stochastic gradient descent (SGD Regressor in sklearn), with penalty of 'Elastic Net' and 'Ridge Regression' as 2 separate cases, K-cross validation with K as 5, find out and compare the output accuracy of applying Linear regression on the California Housing Dataset. Make sure to use Standard scaler on the dataset.


In [2]:
# --- Requirements ---
# pandas, numpy, scikit-learn

import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error

# 1) Load dataset as pandas DataFrame
data = fetch_california_housing(as_frame=True)
df = data.frame
X = df.drop(columns=["MedHouseVal"])
y = df["MedHouseVal"]

# 2) Train/Test split (80/20) – split BEFORE any scaling
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# 3) CV strategy for GridSearch (on TRAIN only)
inner_cv = KFold(n_splits=5, shuffle=True, random_state=42)

# 4) Common SGD settings to keep things stable
common = dict(max_iter=5000, tol=1e-4, random_state=42)

# Helper to evaluate best model on test
def evaluate_best(name, gs, X_test, y_test):
    y_pred = gs.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    print(f"\n[{name}] Best params: {gs.best_params_}")
    print(f"[{name}] Best CV R²: {gs.best_score_:.4f}")
    print(f"[{name}] Test  R²:   {r2:.4f}")
    print(f"[{name}] Test  RMSE: {rmse:.4f}")
    return {"Model": name, "Best CV R²": gs.best_score_, "Test R²": r2, "Test RMSE": rmse}

# --------------------------
# A) LINEAR REGRESSION (no penalty)
# --------------------------
linear_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("sgd", SGDRegressor(penalty=None, **common))
])

# For pure Linear (penalty=None), tune learning rate schedule + step size.
linear_grid = {
    "sgd__learning_rate": ["constant", "invscaling", "adaptive"],
    "sgd__eta0": [1e-4, 1e-3, 1e-2]     # initial step size
}

linear_gs = GridSearchCV(
    estimator=linear_pipe,
    param_grid=linear_grid,
    scoring="r2",
    cv=inner_cv,
    n_jobs=-1,
    refit=True,
    verbose=0
)
linear_gs.fit(X_train, y_train)

# --------------------------
# B) RIDGE REGRESSION (L2)
# --------------------------
ridge_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("sgd", SGDRegressor(penalty="l2", **common))
])

ridge_grid = {
    "sgd__alpha": [1e-4, 1e-3, 1e-2, 1e-1],           # regularization strength
    "sgd__learning_rate": ["constant", "invscaling", "adaptive"],
    "sgd__eta0": [1e-4, 1e-3, 1e-2]
}

ridge_gs = GridSearchCV(
    estimator=ridge_pipe,
    param_grid=ridge_grid,
    scoring="r2",
    cv=inner_cv,
    n_jobs=-1,
    refit=True,
    verbose=0
)
ridge_gs.fit(X_train, y_train)

# --------------------------
# C) ELASTIC NET (L1 + L2)
# --------------------------
elastic_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("sgd", SGDRegressor(penalty="elasticnet", **common))
])

elastic_grid = {
    "sgd__alpha": [1e-4, 1e-3, 1e-2, 1e-1],
    "sgd__l1_ratio": [0.15, 0.3, 0.5, 0.7, 0.85],
    "sgd__learning_rate": ["constant", "invscaling", "adaptive"],
    "sgd__eta0": [1e-4, 1e-3, 1e-2]
}

elastic_gs = GridSearchCV(
    estimator=elastic_pipe,
    param_grid=elastic_grid,
    scoring="r2",
    cv=inner_cv,
    n_jobs=-1,
    refit=True,
    verbose=0
)
elastic_gs.fit(X_train, y_train)

# 5) Evaluate all best models on the TEST set (20%) and summarize
rows = []
rows.append(evaluate_best("Linear (SGD, no penalty)", linear_gs, X_test, y_test))
rows.append(evaluate_best("Ridge (SGD, L2)", ridge_gs, X_test, y_test))
rows.append(evaluate_best("Elastic Net (SGD)", elastic_gs, X_test, y_test))

summary = pd.DataFrame(rows)
print("\n=== Summary (Best models on 20% Test) ===")
print(summary.to_string(index=False))



[Linear (SGD, no penalty)] Best params: {'sgd__eta0': 0.0001, 'sgd__learning_rate': 'adaptive'}
[Linear (SGD, no penalty)] Best CV R²: 0.6113
[Linear (SGD, no penalty)] Test  R²:   0.5756
[Linear (SGD, no penalty)] Test  RMSE: 0.7458

[Ridge (SGD, L2)] Best params: {'sgd__alpha': 0.0001, 'sgd__eta0': 0.0001, 'sgd__learning_rate': 'adaptive'}
[Ridge (SGD, L2)] Best CV R²: 0.6113
[Ridge (SGD, L2)] Test  R²:   0.5756
[Ridge (SGD, L2)] Test  RMSE: 0.7457

[Elastic Net (SGD)] Best params: {'sgd__alpha': 0.001, 'sgd__eta0': 0.0001, 'sgd__l1_ratio': 0.85, 'sgd__learning_rate': 'adaptive'}
[Elastic Net (SGD)] Best CV R²: 0.6113
[Elastic Net (SGD)] Test  R²:   0.5765
[Elastic Net (SGD)] Test  RMSE: 0.7450

=== Summary (Best models on 20% Test) ===
                   Model  Best CV R²  Test R²  Test RMSE
Linear (SGD, no penalty)    0.611292 0.575562   0.745780
         Ridge (SGD, L2)    0.611286 0.575605   0.745742
       Elastic Net (SGD)    0.611293 0.576498   0.744957


The experiment compared three stochastic gradient descent (SGD)–based linear models — Linear Regression (no penalty), Ridge Regression (L2 penalty), and Elastic Net (L1 + L2 penalties) — using the California Housing Dataset.
All models were standardized and evaluated with 5-fold cross-validation and a 20% test split.}

These results indicate that regularization had a minimal effect on model accuracy for this dataset.
The Elastic Net model slightly outperformed the others with the lowest RMSE (0.7449) and highest R² (0.576), suggesting a very small improvement in generalization.